Name & ID : Muhammad Haziq bin Abdullah (SW01083756), Mohamad Naqib bin Mustapa (SW01083743)


In [39]:
import pandas as pd
import re
import nltk
from textblob import TextBlob

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [40]:
df = pd.read_csv("Reviews.csv")

In [41]:
df = df[['Score', 'Text']].dropna()

In [42]:
df = df[df['Score'] != 3]

In [43]:
df['Label'] = df['Score'].apply(lambda x: 1 if x >= 4 else 0)

In [44]:
pos_df = df[df['Label'] == 1].sample(n=10000, random_state=42)
neg_df = df[df['Label'] == 0].sample(n=10000, random_state=42)
df = pd.concat([pos_df, neg_df]).sample(frac=1, random_state=42)

In [45]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [46]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [47]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]
    
    return ' '.join(words)

df['CleanText'] = df['Text'].apply(preprocess_text)

In [48]:
X_train, X_test, y_train, y_test = train_test_split(
    df['CleanText'],
    df['Label'],
    test_size=0.2,
    random_state=42,
    stratify=df['Label']
)

In [49]:
# -----------------------------
# 1. Lexicon-Based Method
# -----------------------------
lexicon_pred = []
for text in X_test:
    polarity = TextBlob(text).sentiment.polarity
    lexicon_pred.append(1 if polarity >= 0 else 0)

print("Lexicon-Based Method")
print("Accuracy:", accuracy_score(y_test, lexicon_pred))
print(classification_report(y_test, lexicon_pred))

Lexicon-Based Method
Accuracy: 0.6575
              precision    recall  f1-score   support

           0       0.89      0.36      0.51      2000
           1       0.60      0.96      0.74      2000

    accuracy                           0.66      4000
   macro avg       0.75      0.66      0.62      4000
weighted avg       0.75      0.66      0.62      4000



In [50]:
# -----------------------------
# 2. Bag-of-Words + Linear SVM
# -----------------------------
bow_vectorizer = CountVectorizer(
    stop_words='english',
    lowercase=True,
    token_pattern=r'(?u)\b[a-zA-Z]{2,}\b',
    max_features=5000
)

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

bow_model = LinearSVC()
bow_model.fit(X_train_bow, y_train)
bow_pred = bow_model.predict(X_test_bow)

print("Bag-of-Words + Linear SVM")
print("Accuracy:", accuracy_score(y_test, bow_pred))
print(classification_report(y_test, bow_pred))

Bag-of-Words + Linear SVM
Accuracy: 0.83075
              precision    recall  f1-score   support

           0       0.84      0.82      0.83      2000
           1       0.82      0.84      0.83      2000

    accuracy                           0.83      4000
   macro avg       0.83      0.83      0.83      4000
weighted avg       0.83      0.83      0.83      4000



C:\Users\user\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [51]:
# -----------------------------
# 3. TF-IDF + Linear SVM
# -----------------------------
tfidf_vectorizer = TfidfVectorizer(
    stop_words='english',
    lowercase=True,
    token_pattern=r'(?u)\b[a-zA-Z]{2,}\b',
    max_features=5000
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

tfidf_model = LinearSVC()
tfidf_model.fit(X_train_tfidf, y_train)
tfidf_pred = tfidf_model.predict(X_test_tfidf)

print("TF-IDF + Linear SVM")
print("Accuracy:", accuracy_score(y_test, tfidf_pred))
print(classification_report(y_test, tfidf_pred))

TF-IDF + Linear SVM
Accuracy: 0.8615
              precision    recall  f1-score   support

           0       0.86      0.86      0.86      2000
           1       0.86      0.86      0.86      2000

    accuracy                           0.86      4000
   macro avg       0.86      0.86      0.86      4000
weighted avg       0.86      0.86      0.86      4000



(1)The lexicon-based method had the weakest performance because it depends only on predefined word polarity and struggles with context, sarcasm, product-specific language, and mixed opinions.
(2)The BoW + Linear SVM model performed much better because it learned patterns from the dataset directly.
(3)The TF-IDF + Linear SVM model achieved the best result because TF-IDF reduced the impact of overly common words and highlighted more meaningful terms for sentiment classification.

Among the three methods, TF-IDF + Linear SVM is the best model for sentiment classification on Reviews.csv because it achieved the highest accuracy and the most balanced overall performance. The lexicon-based approach works only as a basic baseline, while BoW + Linear SVM is strong but slightly weaker than TF-IDF. Therefore, the most suitable method for this dataset is TF-IDF with a machine learning classifier.

In [52]:
df[['CleanText', 'Label']].to_csv("Reviews_extracted.csv", index=False)